<a href="https://colab.research.google.com/github/emijlop/master_thesis/blob/main/BERT_research_note_eml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Dataset packages
!pip install -q -U datasets bertopic

In [ ]:
from datasets import load_dataset,  get_dataset_config_names
from umap import UMAP
import pandas as pd                                            # work with tables
from sklearn.feature_extraction.text import CountVectorizer    # word counting inside BERTopic
from bertopic import BERTopic                                  # the topic model

pd.set_option('display.max_colwidth', None)   # show full sentences in tables
pd.set_option('display.max_columns', None)    # show all columns

In [ ]:
def load_bbq(subset): # load the bbq subset from HF as a pandas table
    ds = load_dataset(
        "parquet",
        data_files=f"hf://datasets/heegyu/bbq@refs%2Fconvert%2Fparquet/{subset}/test/*.parquet",
        split="train",
    )
    return ds.to_pandas()

df = pd.concat(
    [load_bbq("Gender_identity"), load_bbq("Sexual_orientation")],
    ignore_index=True,
)
print(df.shape)
print(df["category"].value_counts)

(6536, 13)
<bound method IndexOpsMixin.value_counts of 0          Gender_identity
1          Gender_identity
2          Gender_identity
3          Gender_identity
4          Gender_identity
               ...        
6531    Sexual_orientation
6532    Sexual_orientation
6533    Sexual_orientation
6534    Sexual_orientation
6535    Sexual_orientation
Name: category, Length: 6536, dtype: object>


In [ ]:
df = df[df["context_condition"] == "disambig"]   # keep the full scenarios
df = df.drop_duplicates("context")               # each context appears twice (two questions)
df = df.reset_index(drop=True)                   # renumber rows 0, 1, 2, ...

print(df.shape)
print(df["category"].value_counts())

(1634, 13)
category
Gender_identity       1418
Sexual_orientation     216
Name: count, dtype: int64


In [ ]:
TAG_MAP = {"man": "M", "boy": "M", "woman": "F", "girl": "F"}

def group_pair(ai):
    """Turn answer_info into a label like 'F vs M'."""
    tags = sorted(TAG_MAP.get(v[1], v[1])
                  for v in ai.values() if v[1] != "unknown")
    return " vs ".join(tags)

df["groups"] = df["answer_info"].apply(group_pair)

df["groups"].value_counts()

,count
groups,
F vs M,1198
nonTrans vs trans,116
nonTrans_M vs trans_F,50
nonTrans_F vs trans_M,50
bisexual vs gay,38
gay vs pansexual,30
bisexual vs lesbian,28
gay vs lesbian,26
gay vs straight,26


In [ ]:
docs = df["context"].tolist()
len(docs)

1634

In [ ]:
# Configure
topic_model = BERTopic(             # (defaults)
    # 3. Clustering
    min_topic_size = 10,            # (10) important! equivalent to HDBSCAN min_cluster_size

    # 4. Tokenizer/Vectorizer       # (CountVectorizer())
    vectorizer_model = CountVectorizer(stop_words="english", min_df=1), # words must appear in at least 1 topic

    # 5. cTF-IDF -> Topic-Terms
    top_n_words = 10,               # (10) increase for more detailed topic descriptions
    #nr_topics = None,               # (None) heed warnings!
)
# Train
topics, probs = topic_model.fit_transform(docs) # topics, probs may get updated later so we later use "live" attributes instead!
print(f'Initial number of topics (incl. -1): {topic_model.get_topic_info().shape[0]}')
print(f'Initial vocabulary size: {len(topic_model.vectorizer_model.get_feature_names_out())}\n')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Initial number of topics (incl. -1): 43
Initial vocabulary size: 713



In [ ]:
#| code-summary: 'Display topic embeddings as output of Step #3: Clustering'
display(topic_model.topic_embeddings_.shape)
pd.DataFrame(
    topic_model.topic_embeddings_,
    index = topic_model.get_topic_info()['Topic'],
).head()

(43, 384)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299,300,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,326,327,328,329,330,331,332,333,334,335,336,337,338,339,340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383
Topic,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
-1,0.047521,0.014230,-0.001799,0.030416,0.047842,-0.034623,0.022065,-0.021189,-0.005503,-0.025202,0.030094,0.014833,-0.034311,0.009235,0.040710,-0.006805,-0.012822,-0.001447,-0.038018,0.033126,-0.030649,-0.042006,0.011180,-0.043031,-0.008656,-0.019709,0.039790,0.009523,-0.012210,0.065582,-0.040586,0.039038,-0.048319,0.027430,-0.067864,-0.004709,0.020858,-0.019196,0.067292,-0.009597,-0.000229,-0.008247,0.042558,-0.060848,-0.026577,0.029407,0.062510,-0.019518,-0.043621,-0.056162,0.031190,-0.025145,0.007207,-0.013165,0.005490,0.031465,-0.015633,-0.001037,0.009740,0.011257,0.012299,-0.030759,-0.018576,0.012837,-0.035779,0.031630,-0.037594,0.043409,0.064614,-0.033700,-0.011798,0.048197,-0.029121,0.016428,0.003697,-0.020224,0.017986,0.057552,0.025424,-0.041900,-0.015867,-0.027716,0.012704,-0.049668,-0.002588,-0.015929,0.005567,-0.017594,-0.014272,0.024775,-0.032648,0.053221,0.051486,-0.067606,0.028781,-0.013837,-0.051663,-0.005072,-0.024262,0.047442,-0.033551,0.033192,0.005552,0.062012,-0.010357,0.027940,-0.002472,-0.050536,-0.028202,-0.011291,-0.001650,-0.040009,0.064674,-0.029905,-0.036397,0.021350,0.019836,0.017733,0.055635,0.016931,-0.065241,-0.016615,0.030326,-0.011494,-0.019071,-0.003689,0.006253,2.997290e-33,-0.022794,0.003301,-0.063662,-0.020707,0.008206,0.005588,0.011268,-0.008362,0.015643,-0.008920,0.016683,0.032749,0.035146,-0.008691,-0.018124,-0.012816,0.016800,0.076104,-0.013915,0.006261,0.034733,0.038464,-0.049910,0.044649,-0.083413,0.014082,-0.003060,0.014382,0.005405,0.017194,0.001931,0.023069,0.014636,0.005564,0.043625,0.029005,0.048322,-0.002024,-0.007290,-0.039676,-0.005915,-0.018437,-0.068381,-0.030662,-0.060421,0.057152,-0.023404,-0.066446,-0.040080,0.008961,-0.022563,0.048866,-0.005529,0.070217,-0.069075,0.014661,0.036672,-0.031447,0.019659,0.007409,0.009361,0.074620,-0.013888,-0.037847,-0.011215,-0.010037,0.046448,-0.025781,0.028663,-0.002805,-0.007412,0.005297,-0.023452,-0.012347,0.024754,-0.066831,-0.050725,-0.027046,0.072740,-0.040571,0.049405,-0.007866,-0.020296,-0.068441,-0.076686,-0.001960,-0.015398,-0.049983,-0.021012,0.024423,0.016798,0.034446,-0.010906,-0.020384,0.045630,-4.423408e-33,-0.036105,-0.015246,-0.008616,-0.02

In [ ]:
#| code-summary: 'Display Topic-Term Matrix of cTF-IDF weights as output of step #5'
display(topic_model.c_tf_idf_.shape)
pd.DataFrame(
    topic_model.c_tf_idf_.toarray(), # can limit [:n]
    index = topic_model.get_topic_info()['Topic'], # can limit [:n]
    columns = topic_model.vectorizer_model.get_feature_names_out(),
).head()


(43, 713)

,12,30,9am,able,absence,abuse,academic,accepted,accessories,accidentally,accusations,acquaintance,active,actually,admitted,adopt,adoption,advanced,advantage,advice,affording,afternoon,agency,aggressive,ago,aids,algorithm,answer,answering,antiretroviral,anxiety,apartment,application,approved,argument,arms,asked,assaulting,assertive,athletes,attendees,attending,awareness,away,babysit,babysitter,bad,badly,bar,barbara,bartender,bartenders,basketball,beginning,behavioral,betty,biggest,biology,bipolar,bisexual,bit,board,bombed,bottle,boy,boys,breakup,brian,bruises,building,bullying,busy,calculated,called,calm,came,campus,candidate,cardio,care,carefully,caring,carol,cars,caught,cause,celebrity,center,challenges,challenging,charges,charles,charless,chatted,cheat,cheated,check,checked,child,children,choice,choices,choosing,chose,chosen,christopher,christophers,cis,cisgender,class,classes,client,clients,clinic,close,clothes,clothing,club,coach,colleague,college,colleges,come,comfort,comfortable,coming,commenting,commitment,company,competing,completely,complex,computer,conference,confessed,confident,connect,consider,considering,consoling,control,convinced,cool,cough,counseling,counselling,counselor,couple,couples,crisis,crying,daniel,daniels,dark,dating,daughter,david,davids,day,dealing,declined,decoration,demeanor,denim,depression,desk,diagnosis,did,didnt,difference,different,difficult,dinner,discriminated,discussing,disorder,distance,distant,districts,divorce,divorced,doctor,doctors,does,doesnt,doing,domestic,donald,donalds,donna,dont,dorothy,drink,dropped,drugs,drunk,dysfunctional,early,easy,ecology,education,edward,edwards,effect,elizabeth,emotional,emotionally,end,ended,engineering,enjoy,enjoyed,entire,escalated,event,exasperated,excited,experience,experienced,experiences,experiments,explanations,extra,face,failing,fairly,faithful,faking,family,fashion,feel,feeling,feelings,feels,felt,female,fifth,fight,fights,film,finally,finish,fit,flirting,fluid,follower,forth,foster,friend,friendly,friends,fully,future,gay,general,generally,george,georges,getting,girl,given,giving,glass,gone,good,gossip,got,gotten,grad,great,greet,group,guardians,guess,gym,hair,hand,hanging,happened,happily,happy,hard,having,head,health,heated,heavy,helen,help,history,hit,hiv,holidays,home,house,houses,husband,husbands,identifies,ids,ignoring,ill,im,immediately,impatient,impressed,inappropriate,including,indecisive,influence,initiative,intense,intention,interested,interns,interruption,interview,interviewed,interviewing,interviews,investing,invitation,invited,isnt,issue,issues,james,jamess,jennifer,job,john,johns,joseph,josephs,jumped,junior,just,karen,karens,kenneth,kenneths,kept,kids,kind,know,knowingly,knows,lab,late,lately,later,leader,learn,learned,leave,leaving,left,lesbian,lesson,let,level,lifelong,light,like,likes,linda,lindas,lisa,listening,little,lives,living,local,long,looked,looking,lost,lot,lower,lunch,major,majors,making,male,man,manages,mans,margaret,margarets,maria,mark,marriage,married,mary,marys,math,meeting,mental,mentioned,met,michael,middle,minutes,missed,mix,monogamous,month,months,moved,moving,music,mutual,nancy,natural,nearby,nearly,needed,needs,negative,neighbor,neighbors,nervous,network,networking,new,newspaper,nice,night,normal,notes,noting,offender,offered,offering,office,old,opening,opportunity,opposite,options,orientation,outside,overalls,painting,pansexual,paper,parent,parents,park,partner,partners,parttime,party,past,patricia,paul,pauls,pending,people,person,personal,phones,physics,physique,pick,picnic,planned,plans,playground,pleasure,plenty,police,politely,position,positive,prefers,prepared,preschool,presented,presenters,presenting,pretty,primary,problem,projects,promised,promotion,proposed,provides,punch,question,questions,quiet,quits,quitting,rational,reach,react,readily,ready,realizing,really,reason,received,recent,recently,recommended,refused,regardless,registry,rejected,relationship,relationships,relies,rely,remain,removed,r

In [ ]:
#| code-summary: 'Display a few per-topic row term sums'
topic_model.c_tf_idf_[0:10].sum(axis=1)

matrix([[2.97769997],
        [2.3770473 ],
        [2.35250499],
        [2.27367336],
        [2.03744487],
        [2.27208064],
        [2.20282741],
        [2.23403878],
        [2.52694066],
        [2.4529135 ]])

In [ ]:
#| code-summary: 'Display the topic summary by document count order'
#| code-fold: false
display(topic_model.get_topic_info().shape)
topic_model.get_topic_info().iloc[:, :-1].head(10) # MODIFY: to see more/less topics (omit Representative_Docs)

(43, 5)

,Topic,Count,Name,Representation
0,-1,44,-1_orientation_sexual_bisexual_registry,"[orientation, sexual, bisexual, registry, neighbors, flirting, gay, identifies, friend, moved]"
1,0,54,0_arms_bruises_counselling_recommended,"[arms, bruises, counselling, recommended, try, seen, mutual, learn, couples, friend]"
2,1,52,1_center_ago_coming_crisis,"[center, ago, coming, crisis, slow, violence, domestic, nervous, able, counseling]"
3,2,52,2_meeting_9am_notes_running,"[meeting, 9am, notes, running, wait, taking, start, board, late, today]"
4,3,52,3_math_class_advanced_middle,"[math, class, advanced, middle, thought, easy, scared, talk, hard, failing]"
5,4,52,4_breakup_consoling_uncomfortably_stood,"[breakup, consoling, uncomfortably, stood, did, difficult, experienced, talking, recently, friend]"
6,5,52,5_questions_asked_algorithm_candidate,"[questions, asked, algorithm, candidate, company, interviewing, behavioral, software, engineering, donna]"
7,6,52,6_teacher_parents_choices_reviews,"[teacher, parents, choices, reviews, presented, yelp, preschool, experience, gotten, negative]"
8,7,52,7_fight_happened_bartenders_serve,"[fight, happened, bartenders, serve, noting, outside, bartender, refused, reporters, sense]"
9,8,52,8_mentioned_semester_early_end,"[mentioned, semester, early, end, doing, opportunity, dropped, fairly, lives, primary]"


In [ ]:
#| code-summary: 'Visualize a list of topics by their word scores'
topic_model.visualize_barchart( # (defaults)
    topics = [0, 1, 2],         # (None), list overrides top_n_topics
    top_n_topics = 8,           # (8)
    n_words = 10,               # (5)
    autoscale = True,           # (False), may not see all words otherwise!
)

In [ ]:
#| code-summary: 'Analyze one topic by its raw word scores'
topic_model.get_topic(0)

[('arms', np.float64(0.12828456054713724)),
 ('bruises', np.float64(0.12828456054713724)),
 ('counselling', np.float64(0.12828456054713724)),
 ('recommended', np.float64(0.12828456054713724)),
 ('try', np.float64(0.12828456054713724)),
 ('seen', np.float64(0.12828456054713724)),
 ('mutual', np.float64(0.12828456054713724)),
 ('learn', np.float64(0.12828456054713724)),
 ('couples', np.float64(0.12240372634677449)),
 ('friend', np.float64(0.10387537946226218))]

In [ ]:
#| code-summary: 'Change topic label names'
#| code-fold: false
topn = 10 # MODIFY?

vectorizer_model = CountVectorizer(stop_words='english', min_df=2, max_df=0.5) # MODIFY: normalization options
topic_model.update_topics(docs, vectorizer_model=vectorizer_model, top_n_words=topn) # MODIFY: n_gram_range?
print(f'Modified vocabulary size: {len(topic_model.vectorizer_model.get_feature_names_out())}')
topic_model.get_topic_info().iloc[:, :-1].head()

Modified vocabulary size: 253


,Topic,Count,Name,Representation
0,-1,44,-1_sexual_bisexual_identifies_gay,"[sexual, bisexual, identifies, gay, feeling, moved, lesbian, friend, partner, right]"
1,0,54,0_couples_responsible_friend_married,"[couples, responsible, friend, married, happened, started, couple, asked, just, linda]"
2,1,52,1_center_able_counseling_week,"[center, able, counseling, week, help, today, experienced, getting, started, pretty]"
3,2,52,2_meeting_wait_taking_start,"[meeting, wait, taking, start, today, late, board, responsible, donna, karen]"
4,3,52,3_math_class_scared_talk,"[math, class, scared, talk, hard, failing, school, stayed, day, moved]"


In [ ]:
#| code-summary: 'Visualize topics in UMAP 2-dimensional space'
topic_model.visualize_topics( # (defaults)
    use_ctfidf = False,       # (False) map embeddings / True: map cTF-IDF weights
    top_n_topics = None,      # (None) all
    custom_labels = True,     # (False)
)

In [ ]:
#| code-summary: "Visualize topic embeddings' cosine similarities and look for clusters of topics"
topic_model.visualize_heatmap( # (defaults)
    use_ctfidf = False,        # (False) map embeddings / True: map cTF-IDF weights
    #n_clusters = 15,           # (None) <- MODIFY: powerful! creates ordered CLUSTERS of similar topics
    top_n_topics = None,       # (None) all
    custom_labels = True,      # (False)
)

In [ ]:
#| code-summary: 'Re-configure and train BERTopic, modifying `min_topic_size`'
#| code-fold: false
# for less topics, try increasing min_topic_size
# for more topics, try decreasing min_topic_size
# Configure
topic_model = BERTopic(             # (defaults)
    # 3. Clustering
    min_topic_size = 25,

    # 4. Tokenizer/Vectorizer       # (CountVectorizer())
    vectorizer_model = CountVectorizer(stop_words="english", min_df=2),

    # 5. cTF-IDF -> Topic-Terms
    top_n_words = 10,               # (10) increase for more detailed topic descriptions
    nr_topics = None,               # (None) heed warnings!
)
# Train
topics, probs = topic_model.fit_transform(docs) # topics, probs may get updated later so we later use "live" attributes instead!
print(f'Modified number of topics (incl. -1): {topic_model.get_topic_info().shape[0]}')
print(f'Modified vocabulary size: {len(topic_model.vectorizer_model.get_feature_names_out())}\n')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Modified number of topics (incl. -1): 31
Modified vocabulary size: 245

